Data Generator

In [ ]:
import numpy as np
import tensorflow as tf
import os
import json

# Load metadata
with open('../../dataset/vocab.json', 'r') as f:
    vocab_data = json.load(f)

VOCAB_SIZE      = vocab_data['vocab_size']
MAX_LENGTH      = vocab_data['max_length']
WORD_TO_INDEX   = vocab_data['word_to_index']
FEATURE_DIR     = "../../../dataset/features"

def data_generator(df, batch_size):
    while True:
        for i in range(0, len(df), batch_size):
            batch_df = df.iloc[i:i+batch_size]
            features_input = []
            captions_input = []
            targets = []

            for _, row in batch_df.iterrows():
                feat_path = os.path.join(FEATURE_DIR, row['image'] + ".npy")
                feature = np.load(feat_path)
                
                # Tokenize Caption
                seq = [WORD_TO_INDEX.get(w, 0) for w in row['caption'].split()]
                
                in_seq = seq[:-1]   # Input sequence: <start> ... SN-1
                out_seq = seq[1:]   # Target sequence: S0 ... <end>
                
                # Padding
                in_seq = tf.keras.preprocessing.sequence.pad_sequences([in_seq], maxlen=MAX_LENGTH)[0]
                out_seq = tf.keras.preprocessing.sequence.pad_sequences([out_seq], maxlen=MAX_LENGTH)[0]
                
                features_input.append(feature)
                captions_input.append(in_seq)
                targets.append(out_seq)
            
            yield [np.array(features_input), np.array(captions_input)], np.array(targets)

Build RNN Model

In [ ]:
def build_rnn_model(num_layers, hidden_size, embed_dim=256):
    # Input 1: CNN Feature
    image_input = tf.keras.Input(shape=(2048,), name="image_input")
    image_emb = tf.keras.layers.Dense(embed_dim, activation='relu')(image_input)
    image_emb = tf.keras.layers.Reshape((1, embed_dim))(image_emb)

    # Input 2: Caption Sequence
    caption_input = tf.keras.Input(shape=(MAX_LENGTH,), name="caption_input")
    caption_emb = tf.keras.layers.Embedding(VOCAB_SIZE, embed_dim, mask_zero=True)(caption_input)

    # Pre-inject
    merged = tf.keras.layers.Concatenate(axis=1)([image_emb, caption_emb])

    # Recurrent Layers
    x = merged
    for i in range(num_layers):
        x = tf.keras.layers.SimpleRNN(hidden_size, return_sequences=True)(x)

    # Output Layer -> timestep teks = index 1 dst.
    x = tf.keras.layers.Lambda(lambda t: t[:, 1:, :])(x)
    output = tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(VOCAB_SIZE, activation='softmax'))(x)

    model = tf.keras.Model(inputs=[image_input, caption_input], outputs=output)
    model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

Training Loop

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Data Preparation
df_captions = pd.read_csv("../../../dataset/captions.txt")
df_captions['caption'] = "<start> " + df_captions['caption'].str.lower() + " <end>"
train_df, val_df = train_test_split(df_captions, test_size=0.1, random_state=42)

# Hyperparameters
layers_variations = [1, 2, 3]
hidden_variations = [256, 512]

for n_layers in layers_variations:
    for h_size in hidden_variations:
        version_name = f"rnn_l{n_layers}_h{h_size}"
        save_path = f"../weights/{version_name}/"
        os.makedirs(save_path, exist_ok=True)
        
        print(f"\nTraining Variasi: {version_name}")
        
        model = build_rnn_model(num_layers=n_layers, hidden_size=h_size)
        
        # Training
        batch_size = 64
        model.fit(
            data_generator(train_df, batch_size),
            steps_per_epoch=len(train_df) // batch_size,
            validation_data=data_generator(val_df, batch_size),
            validation_steps=len(val_df) // batch_size,
            epochs=32
        )
        
        # Save Model Weights
        model.save_weights(os.path.join(save_path, f"{version_name}_weights.h5"))
        print(f"Bobot disimpan di {save_path}")